In [ ]:
!pip install numba

## ***Normal Case***

In [ ]:
import numpy as np
from numba import cuda

@cuda.jit
def reduce_kernel(input_arr, output_arr):
    # Shared memory
    sdata = cuda.shared.array(1024, dtype=numba.int32)

    tid = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tid

    # Load data into shared memory
    if i < input_arr.size:
        sdata[tid] = input_arr[i]
    else:
        sdata[tid] = 0
    cuda.syncthreads()

    # Reduction (tree pattern)
    stride = 1
    while stride < cuda.blockDim.x:
        if tid % (2 * stride) == 0:
            sdata[tid] += sdata[tid + stride]
        stride *= 2
        cuda.syncthreads()

    # Write result
    if tid == 0:
        output_arr[cuda.blockIdx.x] = sdata[0]

In [ ]:
import numba

# Input
arr = np.array([1,2,3,4,5,6,7,8], dtype=np.int32)

# Device memory
d_input = cuda.to_device(arr)
d_output = cuda.device_array(1, dtype=np.int32)

# Launch kernel
threads_per_block = 8
blocks = 1

reduce_kernel[blocks, threads_per_block](d_input, d_output)

# Copy result back
result = d_output.copy_to_host()

print("Final Sum:", result[0])

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Final Sum: 36


## ***Handle odd number case and take input from user***

In [ ]:
import numpy as np
import numba
from numba import cuda

@cuda.jit
def reduce_kernel(input_arr, output_arr, n):
    # Dynamic shared memory
    sdata = cuda.shared.array(shape=0, dtype=numba.int32)

    tid = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tid

    # Load elements safely
    if i < n:
        sdata[tid] = input_arr[i]
    else:
        sdata[tid] = 0
    cuda.syncthreads()

    # Tree-based reduction
    stride = 1
    while stride < cuda.blockDim.x:
        index = 2 * stride * tid
        if index < cuda.blockDim.x and index + stride < n:
            sdata[index] += sdata[index + stride]
        stride *= 2
        cuda.syncthreads()

    # Write block result
    if tid == 0:
        output_arr[cuda.blockIdx.x] = sdata[0]

In [ ]:
# Take input from user
user_input = input("Enter numbers separated by space: ")
arr = np.array(list(map(int, user_input.split())), dtype=np.int32)

n = len(arr)

# GPU memory
d_input = cuda.to_device(arr)

threads_per_block = 1
while threads_per_block < n:
    threads_per_block *= 2   # next power of 2

blocks = 1

d_output = cuda.device_array(blocks, dtype=np.int32)

# Launch kernel (dynamic shared memory)
reduce_kernel[blocks, threads_per_block, 0, threads_per_block * 4](d_input, d_output, n)

# Get result
result = d_output.copy_to_host()

print("Final Sum:", result[0])

Enter numbers separated by space: 2 34 5 67 5 43 2 1
Final Sum: 159


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


## ***Take numbers randomly with user giving number of values in input***

In [ ]:
import numpy as np
import numba
from numba import cuda
import random

@cuda.jit
def reduce_kernel(input_arr, output_arr, n):
    sdata = cuda.shared.array(shape=0, dtype=numba.int32)

    tid = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tid

    # Load into shared memory safely
    if i < n:
        sdata[tid] = input_arr[i]
    else:
        sdata[tid] = 0
    cuda.syncthreads()

    # Reduction (tree pattern)
    stride = 1
    while stride < cuda.blockDim.x:
        index = 2 * stride * tid
        if index < cuda.blockDim.x and index + stride < cuda.blockDim.x:
            sdata[index] += sdata[index + stride]
        stride *= 2
        cuda.syncthreads()

    # Write result
    if tid == 0:
        output_arr[cuda.blockIdx.x] = sdata[0]


#  User input (number of elements)
n = int(input("Enter number of elements: "))

# Generate random numbers
arr = np.array([random.randint(1, 10) for _ in range(n)], dtype=np.int32)

print("Generated array:", arr)

# GPU memory
d_input = cuda.to_device(arr)

# Next power of 2 for threads
threads_per_block = 1
while threads_per_block < n:
    threads_per_block *= 2

blocks = 1
d_output = cuda.device_array(blocks, dtype=np.int32)

# Launch kernel
reduce_kernel[blocks, threads_per_block, 0, threads_per_block * 4](d_input, d_output, n)

# Result
result = d_output.copy_to_host()

print("Final Sum:", result[0])

Enter number of elements: 1024
Generated array: [ 9  1 10 ...  2  6  3]
Final Sum: 5669


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


### ***Multibox reduction***

In [ ]:
import numpy as np
import numba
from numba import cuda
import random
import math

@cuda.jit
def reduce_kernel(input_arr, output_arr, n):
    sdata = cuda.shared.array(shape=0, dtype=numba.int32)

    tid = cuda.threadIdx.x
    i = cuda.blockIdx.x * cuda.blockDim.x + tid

    # Load into shared memory
    if i < n:
        sdata[tid] = input_arr[i]
    else:
        sdata[tid] = 0
    cuda.syncthreads()

    # Reduction within block
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tid < stride:
            sdata[tid] += sdata[tid + stride]
        stride //= 2
        cuda.syncthreads()

    # Write result of block
    if tid == 0:
        output_arr[cuda.blockIdx.x] = sdata[0]


# User input
n = int(input("Enter number of elements: "))

# Random input
arr = np.array([random.randint(1, 10) for _ in range(n)], dtype=np.int32)

print("Generated array (first 20 elements):", arr[:20])

# Copy to GPU
d_input = cuda.to_device(arr)

threads_per_block = 256

# Multi-pass reduction
while True:
    blocks = math.ceil(n / threads_per_block)

    d_output = cuda.device_array(blocks, dtype=np.int32)

    # Launch kernel
    reduce_kernel[blocks, threads_per_block, 0, threads_per_block * 4](d_input, d_output, n)

    if blocks == 1:
        break

    # Prepare for next iteration
    d_input = d_output
    n = blocks

# Final result
result = d_output.copy_to_host()

print("Final Sum:", result[0])

Enter number of elements: 10345
Generated array (first 20 elements): [ 5  7  1  8  4 10  2  6  9  2  2  3  4 10  4 10  7  8  3  1]
Final Sum: 56825


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 41 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
